# MapBiomas Collection 10, Santa Catarina window, for the study-area figure

Exports the MapBiomas land use and land cover map of one year (Collection 10, the collection used as input of the production run) for the Santa Catarina window, at the native 30 m grid, to Google Drive. Run the cells in order in Google Colab, download `brazil_coverage_<year>_SC.tif` from Google Drive and run, from the repository root,

`python analysis/v10/maps/extract_mapbiomas_sc.py brazil_coverage_<year>_SC.tif <year>`

then `FIG1_YEAR=<year> python analysis/v10/maps/study_area_map.py` and `FIG1_YEAR=<year> python analysis/v10/maps/landcover_mesoregion.py`. The 2024 files in `data/external` were made this way on 16 November 2026; aggregated as in the run, the exported 2024 map reproduces the growth-form fractions of `data/run_v10/2024/inputs/growthform_2024.npz` (tree fraction r = 0.99999, largest cell difference 0.006).

In [ ]:
# 1. Parameters
PROJECT = 'your-ee-project'   # <-- your Earth Engine Cloud project
YEAR = 2024
TARGET_COLLECTION = 10        # MapBiomas Collection 10, the collection of the production run
NAME = f'brazil_coverage_{YEAR}_SC'
# Santa Catarina extremes from the IBGE 2024 municipal mesh, with a 0.05 degree margin
BBOX = [-53.8372 - 0.05, -29.3551 - 0.05, -48.3279 + 0.05, -25.9558 + 0.05]

In [ ]:
# 2. Earth Engine
import ee
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)
print('Earth Engine initialised, project', PROJECT)

In [ ]:
# 3. Collections available for the year, and the one used here
col = ee.ImageCollection('projects/mapbiomas-public/assets/brazil/lulc/v1').filter(ee.Filter.eq('year', YEAR))
info = col.aggregate_array('collection_id').getInfo(), col.aggregate_array('version').getInfo()
print('collection_id of the images of', YEAR, ':', info[0], ' version:', info[1])
ids = sorted({float(i) for i in info[0] if int(float(i)) == TARGET_COLLECTION})
assert ids, f'no image of Collection {TARGET_COLLECTION} for {YEAR}'
CID = ids[-1]   # newest release within the collection, as the run code takes the newest available
sel = col.filter(ee.Filter.eq('collection_id', CID))
if sel.size().getInfo() == 0:   # the property may be stored as an integer
    sel = col.filter(ee.Filter.eq('collection_id', int(CID)))
img = ee.Image(sel.first()).select('classification').toByte()
print('using:', ee.Dictionary({'collection_id': img.get('collection_id'), 'version': img.get('version')}).getInfo())

In [ ]:
# 4. Export the Santa Catarina window on the native 30 m grid to Google Drive
region = ee.Geometry.Rectangle(BBOX)
p = img.projection().getInfo()
task = ee.batch.Export.image.toDrive(image=img.clip(region), description=NAME, fileNamePrefix=NAME, region=region,
                                     crs=p['crs'], crsTransform=p['transform'], maxPixels=1e10, fileFormat='GeoTIFF')
task.start()
print('export started:', NAME + '.tif')

In [ ]:
# 5. Status (run again until COMPLETED)
print(task.status()['state'])